# Домашняя работа: подготовка данных — ML Base

Исследуем **Adult**, подготовим признаки и обучим **логистическую регрессию
(`LogisticRegression`)**. В этой версии восемь заданий:

- A — знакомство с данными;
- B — два новых признака;
- C — ограничение крайних значений;
- D — флаги пропусков;
- E — target encoding;
- F — заполнение пропусков и кодирование;
- G — отбор признаков через SelectKBest;
- H — сборка и отправка своей модели в контест.

CSV и архив с ноутбуком доступны в [задаче A контеста](https://contest.yandex.ru/contest/101854/problems/A/). Сохраните `data/adult_train.csv`
и `data/adult_valid.csv` в папку `data` рядом с ноутбуком.

Используйте готовое разбиение train/validation. Все статистики и преобразования
обучайте только на train. В модели задавайте `random_state=42`.
Для итоговой модели используйте только `LogisticRegression`.

Заполняйте `TODO` по порядку и запускайте проверки под заданиями.
В G заполните создание, обучение и применение SelectKBest;
код сравнения моделей уже готов. В H самостоятельно соберите признаки,
предобработку и обучение в две функции для отправки в контест.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from narwhals import DataFrame
from sklearn.metrics import roc_auc_score
from IPython.display import display

# Для случайных экспериментов с NumPy.
np.random.seed(42)

## Adult

Теперь вы пройдете путь от анализа признаков до постепенного улучшения качества модели путем работы с ними самостоятельно. В Adult - датасете для бинарной классификации - одна строка описывает человека. Попробуем предсказать `income_gt_50k`: 1 означает годовой доход выше 50 тысяч долларов, 0 — не выше. Какие признаки, на ваш взгляд, могут помочь?

| Признак | Смысл |
|---|---|
| `age` | Возраст, годы |
| `workclass` | Тип занятости или работодателя |
| `education`, `education_num` | Уровень образования: название и его порядковый номер |
| `occupation` | Профессиональная группа |
| `relationship` | Положение человека в домохозяйстве |
| `capital_gain`, `capital_loss` | Доходы и потери от капитала, доллары |
| `hours_per_week` | Число рабочих часов в неделю |
| `native_country` | Страна происхождения |

Для оценки возьмём **ROC-AUC**: она показывает, насколько хорошо модель ставит людей с доходом >50K выше остальных по предсказанной вероятности. Чем больше, тем лучше; 0.5 соответствует случайному ранжированию, 1 — идеальному.

In [2]:
adult_train = pd.read_csv('data/adult_train.csv')
adult_valid = pd.read_csv('data/adult_valid.csv')
X_adult, y_adult = adult_train.drop(columns='income_gt_50k'), adult_train['income_gt_50k']
X_adult_valid, y_adult_valid = adult_valid.drop(columns='income_gt_50k'), adult_valid['income_gt_50k']
adult_train.head()

,age,workclass,education,education_num,occupation,relationship,capital_gain,capital_loss,hours_per_week,native_country,income_gt_50k
0,56,Private,11th,7,Transport-moving,Husband,0,0,40,United-States,0
1,38,Private,Bachelors,13,Other-service,Own-child,0,0,35,United-States,0
2,18,Private,12th,8,Sales,Own-child,0,0,20,United-States,0
3,25,Private,HS-grad,9,Transport-moving,Own-child,0,0,70,United-States,0
4,52,Private,HS-grad,9,Other-service,Unmarried,0,0,40,India,1


### [A. Познакомимся с данными](https://contest.yandex.ru/contest/101854/problems/A/)

Напишите три небольшие функции:

- `dataset_profile(X, y)` возвращает пять чисел:
  `(n_rows, n_features, n_numeric, n_categorical, positive_rate)`.
  Это число строк, всех признаков, числовых признаков, категориальных признаков
  и доля людей с `y=1`. Числовые колонки — dtype `number`, остальные — категории.
  Target в X не входит; индексы X и y совпадают.
- `missing_report(X)` возвращает DataFrame с колонками `missing_count`,
  `missing_share` и индексом из названий колонок X.
  Доля пропусков считается от всех строк.
- `numeric_summary(X, columns)` возвращает DataFrame с колонками
  `count`, `mean`, `median`, `zero_share` и индексом `columns`.
  Долю нулей считайте **среди известных значений**.

В заготовках замените `None` результатами вычислений.
Сохраните указанный порядок значений и колонок.
Если известных значений нет, счётчик равен 0, остальные статистики — NaN.

**Подсказки.** Размер таблицы — `X.shape`, числовые колонки —
`X.select_dtypes('number')`. Для пропусков пригодятся `isna().sum()` и
`isna().mean()`. Среднее target из нулей и единиц равно доле единиц.
Для числовой сводки сначала выберите `numbers = X[columns]`:
у него есть методы `count()`, `mean()` и `median()`.
Долю нулей можно получить как `numbers.eq(0).sum() / numbers.count()`.

Примените функции к Adult и посмотрите на готовые гистограммы.
Запишите, где есть пропуски и у какого признака много нулей.
Почему ноль в `capital_gain` не стоит считать пропуском?

In [3]:
def dataset_profile(X, y):
    # TODO: n_rows, n_features, n_numeric, n_categorical, positive_rate.
    n_rows = X.shape[0]
    n_features = X.shape[1]
    n_numeric = len(X.select_dtypes("number"))
    n_categorical = len(X.select_dtypes("object"))
    positive_rate = y.eq(1) / n_rows
    return n_rows, n_features, n_numeric, n_categorical, positive_rate


def missing_report(X):
    # TODO: замените None статистиками для каждой колонки X.
    return pd.DataFrame({
        'missing_count': X.isna().sum(),
        'missing_share': X.isna().mean(),
    }, index=X.columns)


def numeric_summary(X, columns):
    # TODO: замените None статистиками для каждой колонки из columns.
    return pd.DataFrame({
        'count': X[columns].count(),
        'mean': X[columns].mean(),
        'median': X[columns].median(),
        'zero_share': X[columns].eq(0).sum() / X[columns].count(),
    }, index=columns)

In [4]:
sample = pd.DataFrame({'value': [0., 2., 4., np.nan], 'group': ['a', 'b', 'a', None]})
labels = pd.Series([0, 1, 1, 0])
assert dataset_profile(sample, labels) == (4, 2, 1, 1, 0.5)
report = missing_report(sample)
assert report['missing_count'].tolist() == [1, 1]
assert np.allclose(report['missing_share'], [0.25, 0.25])
assert np.allclose(
    numeric_summary(sample, ['value']).loc['value'], [3, 2, 2, 1/3]
)

C:\Users\PC\AppData\Local\Temp\ipykernel_10484\2769785993.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  n_categorical = len(X.select_dtypes("object"))


AssertionError: 

In [ ]:
display(dataset_profile(X_adult, y_adult))
display(missing_report(X_adult))
display(numeric_summary(X_adult, ['age', 'hours_per_week', 'capital_gain']))
X_adult[['age', 'hours_per_week', 'capital_gain']].hist(bins=30, figsize=(10, 3))
plt.tight_layout()
plt.show()
# Запишите: где есть пропуски, где длинный хвост, где много нулей.

**Ваши наблюдения:** ...

### [B. Добавим два простых признака](https://contest.yandex.ru/contest/101854/problems/B/)

Логистическая регрессия использует линейную комбинацию входных признаков.
Ей может помочь другое представление дохода от капитала:

- `has_capital_gain` — 1, если `capital_gain > 0`, иначе 0;
- `capital_gain_log` — `np.log1p(capital_gain)`, то есть `log(1 + capital_gain)`.
  Этот признак уменьшает различие между очень большими значениями и сохраняет нули.

Заполните две строки в `add_adult_features(X)`. Функция возвращает копию X
с исходными колонками и этими двумя новыми признаками.
Если `capital_gain` пропущен, оба новых значения тоже должны быть NaN.
Известные суммы неотрицательны.

**Подсказки.** Флаг можно получить как `(gain > 0).astype(float)`.
Чтобы вернуть пропуски, добавьте `.where(gain.notna(), np.nan)`.
`np.log1p` сохраняет NaN сам. Исходный DataFrame менять не нужно:
строка с `copy()` уже написана.

In [ ]:
def add_adult_features(X):
    out = X.copy()
    gain = X['capital_gain']
    out['has_capital_gain'] = None  # TODO: 0/1, при пропуске — NaN.
    out['capital_gain_log'] = None  # TODO: log(1 + gain).
    return out

In [ ]:
sample = pd.DataFrame({'capital_gain': [0., 9., np.nan]}, index=[8, 2, 5])
result = add_adult_features(sample)
assert np.allclose(result['has_capital_gain'], [0., 1., np.nan], equal_nan=True)
assert np.allclose(result['capital_gain_log'], [0., np.log(10), np.nan], equal_nan=True)
assert result.index.equals(sample.index)
assert sample.columns.tolist() == ['capital_gain']

In [ ]:
adult_features = add_adult_features(X_adult)
adult_features[['capital_gain', 'capital_gain_log']].hist(bins=30, figsize=(9, 3))
plt.tight_layout()
plt.show()
# Как изменился масштаб и вид распределения?

### [C. Ограничим крайние значения](https://contest.yandex.ru/contest/101854/problems/C/)

Очень большие или маленькие значения могут влиять на модель.
Попробуем ограничить `hours_per_week`: значения ниже квантили 0,01 заменим
этой границей, значения выше квантили 0,99 — верхней границей.
Квантиль 0,99 — порог, ниже которого находится примерно 99% значений.
Это эксперимент: крайнее значение не обязательно является ошибкой.

Напишите две функции:

- `fit_clip_bounds(X, columns, lower=0.01, upper=0.99)` находит границы по train.
  Верните DataFrame с индексом `columns` и колонками `lower`, `upper`.
  Пропуски при расчёте не учитывайте; интерполяция квантилей — линейная.
  У полностью пустой колонки обе границы равны NaN.
- `apply_clip(X, bounds)` возвращает копию X с ограниченными значениями
  только в колонках из `bounds.index`. Сохраните остальные колонки,
  пропуски, индекс и порядок строк. Границы заново не вычисляйте.
  Граница NaN означает отсутствие ограничения с этой стороны.

**Подсказки.** Для границ пригодится
`X[columns].quantile(lower, interpolation='linear')` и аналогично `upper`.
Для применения — `clip(lower=bounds['lower'], upper=bounds['upper'], axis=1)`.
Параметр `axis=1` сопоставляет границы с названиями колонок.

В примере проверки возьмём квантили 0,25 и 0,75, чтобы границы было легко
посчитать вручную. На Adult используйте значения по умолчанию.

In [ ]:
def fit_clip_bounds(X, columns, lower=0.01, upper=0.99):
    # TODO: вычислите две Series с квантилями по train.
    return pd.DataFrame({
        'lower': None,
        'upper': None,
    }, index=columns)


def apply_clip(X, bounds):
    out = X.copy()
    columns = bounds.index
    out[columns] = None  # TODO: ограничьте значения по сохранённым границам.
    return out

In [ ]:
sample = pd.DataFrame({'value': [0., 10., 20., 30., 1000., np.nan]})
bounds = fit_clip_bounds(sample, ['value'], lower=0.25, upper=0.75)
expected_bounds = pd.DataFrame({'lower': [10.], 'upper': [30.]}, index=['value'])
pd.testing.assert_frame_equal(bounds, expected_bounds)

new = pd.DataFrame({'value': [-100., 20., 2000., np.nan], 'other': [1, 2, 3, 4]},
                   index=[8, 2, 5, 9])
clipped = apply_clip(new, bounds)
assert np.allclose(clipped['value'], [10., 20., 30., np.nan], equal_nan=True)
assert clipped.index.equals(new.index)
pd.testing.assert_series_equal(clipped['other'], new['other'])
assert new.loc[8, 'value'] == -100.

In [ ]:
clip_bounds = fit_clip_bounds(X_adult, ['hours_per_week'])
X_clip_train = apply_clip(X_adult, clip_bounds)
X_clip_valid = apply_clip(X_adult_valid, clip_bounds)
display(clip_bounds)
pd.DataFrame({
    'до': X_adult['hours_per_week'],
    'после': X_clip_train['hours_per_week'],
}).hist(bins=30, figsize=(9, 3))
plt.tight_layout()
plt.show()

Чем ограничение значений отличается от удаления строк?
Почему для validation нельзя вычислять свои границы?

**Ваш ответ:** ...

### [D. Пропуск тоже может быть признаком](https://contest.yandex.ru/contest/101854/problems/D/)

После заполнения пропусков модель не знает, какие значения были известны
изначально. Сохраним эту информацию в отдельных колонках.

Напишите `add_missing_flags(X, columns)`: верните копию X, добавив для каждой
указанной колонки признак `<название>_is_missing`.
Он равен 1, если исходное значение пропущено, и 0 иначе.
Сохраните исходные колонки, индекс и порядок строк.
Новых названий среди исходных колонок нет.

**Подсказка:** `X[col].isna().astype(int)`.
Добавляйте флаги **до заполнения пропусков**.
Для train и validation используйте один и тот же список колонок.

In [ ]:
def add_missing_flags(X, columns):
    out = X.copy()
    for col in columns:
        out[col + '_is_missing'] = None  # TODO: 1 для пропуска, иначе 0.
    return out

In [ ]:
sample = pd.DataFrame({'occupation': ['a', None, 'b'], 'age': [20., np.nan, 40.]},
                      index=[8, 2, 5])
flagged = add_missing_flags(sample, ['occupation', 'age'])
assert flagged['occupation_is_missing'].tolist() == [0, 1, 0]
assert flagged['age_is_missing'].tolist() == [0, 1, 0]
assert pd.api.types.is_integer_dtype(flagged['occupation_is_missing'])
pd.testing.assert_frame_equal(flagged[sample.columns], sample)
assert sample.columns.tolist() == ['occupation', 'age']

In [ ]:
missing_columns = ['occupation', 'workclass', 'native_country']
X_flags_train = add_missing_flags(X_adult, missing_columns)
X_flags_valid = add_missing_flags(X_adult_valid, missing_columns)
flag_columns = [col + '_is_missing' for col in missing_columns]
pd.DataFrame({
    'доля пропусков train': X_flags_train[flag_columns].mean(),
    'доля пропусков validation': X_flags_valid[flag_columns].mean(),
})

Какую информацию теряем, когда просто заменяем пропуск наиболее частым значением?
Что получилось бы, если бы мы создали флаги после заполнения пропусков?

**Ваш ответ:** ...

### [E. Target encoding: средний target для профессии](https://contest.yandex.ru/contest/101854/problems/E/)

Категорию можно заменить средним target в этой категории.
Например, если у пяти человек одной профессии target равен `[0, 1, 0, 1, 0]`,
код этой профессии равен `2 / 5 = 0.4`. Это **target encoding**.

Напишите две функции:

- `fit_target_encoding(categories, y)` получает непустые Series с одинаковыми
  индексами; y содержит 0 и 1 без пропусков. Верните словарь:
  `mapping` — Series со средним target для каждой категории,
  `global_mean` — среднее y по всему train.
- `apply_target_encoding(categories, state)` возвращает вещественную Series
  с кодами, исходным индексом и именем `target_encoding`.
  Неизвестную категорию замените на `global_mean`.
  Для пустого входа верните пустую Series того же формата.

Пропуск считайте отдельной категорией `__MISSING__`: нужные строки уже написаны.
Исходные Series не меняйте.

**Подсказки:** `table.groupby('category')['target'].mean()` строит словарь средних,
`labels.map(state['mapping'])` заменяет категории числами,
`fillna(state['global_mean'])` обрабатывает неизвестные категории.

В этой задаче посчитайте средние **по train** и закодируйте **validation**.
Обучать модель на этих кодах в обязательной части не нужно.
Не используйте target validation для получения кодов.

Почему нужна осторожность с train? Если категория встретилась один раз,
средний target в ней точно равен ответу этой строки. Модель может запомнить
ответ вместо закономерности. Для обучения модели код каждой обучающей строки
следует рассчитывать по другим строкам; такую схему здесь реализовывать не требуется.

In [ ]:
def fit_target_encoding(categories, y):
    labels = categories.astype(object).fillna('__MISSING__')
    table = pd.DataFrame({'category': labels, 'target': y})
    return {
        'mapping': None,  # TODO: средний target по категориям.
        'global_mean': None,  # TODO: средний target по всему train.
    }


def apply_target_encoding(categories, state):
    labels = categories.astype(object).fillna('__MISSING__')
    codes = None  # TODO: map, заполнение неизвестных, astype(float).
    return codes.rename('target_encoding')

In [ ]:
categories = pd.Series(['a', 'a', 'b', None], index=[4, 7, 1, 9])
labels = pd.Series([0, 1, 0, 1], index=categories.index)
target_state = fit_target_encoding(categories, labels)
assert target_state['global_mean'] == 0.5
new_categories = pd.Series(['a', 'b', None, 'unknown'], index=[8, 2, 5, 3])
codes = apply_target_encoding(new_categories, target_state)
expected_codes = pd.Series([0.5, 0., 1., 0.5], index=new_categories.index,
                           name='target_encoding')
pd.testing.assert_series_equal(codes, expected_codes)

In [ ]:
occupation_target_state = fit_target_encoding(X_adult['occupation'], y_adult)
occupation_valid_codes = apply_target_encoding(
    X_adult_valid['occupation'], occupation_target_state
)
display(pd.DataFrame({
    'профессия': X_adult_valid['occupation'],
    'код по train': occupation_valid_codes,
}).head(12))
occupation_target_state['mapping'].sort_values(ascending=False)

Какой код получит неизвестная профессия?
Почему среднее target по двум людям менее надёжно, чем по двум тысячам?
Что произойдёт, если закодировать уникальную категорию собственным target?

**Ваш ответ:** ...

### [F. Подготовим таблицу для модели](https://contest.yandex.ru/contest/101854/problems/F/)

Модель принимает числа без пропусков. Подготовка состоит из двух функций:
`fit_preprocessor(X)` запоминает настройки по train,
`transform_preprocessor(X, state)` применяет их к любой новой таблице.

**В `fit_preprocessor` осталось заполнить только две строки с TODO:**

1. Найдите медианы числовых колонок train: `X[numeric].median()`.
   Для полностью пустой колонки используйте 0 — добавьте `.fillna(0)`.
   Сохраните Series в `medians`. Следующая строка уже заполняет пропуски.
2. Обучите `StandardScaler` на заполненной числовой таблице `numbers`:
   `StandardScaler().fit(numbers)`. Сохраните объект в `scaler`.
   Он запомнит средние и стандартные отклонения train (`ddof=0`);
   для постоянного признака делитель автоматически равен 1.

Остальной код уже написан. Прочитайте, что он делает:

3. В категориях заменяет пропуски на `__MISSING__` и выполняет one-hot через
   `pd.get_dummies`: каждая категория становится отдельной колонкой из 0 и 1.
4. Сохраняет исходные колонки (`input_columns`), списки числовых и категориальных
   (`numeric`, `categorical`), медианы (`medians`), обученный `scaler`
   и итоговые колонки (`columns`) в словарь `state`.
5. В `transform_preprocessor` заполняет пропуски сохранёнными медианами,
   вызывает `scaler.transform`, кодирует категории и приводит результат
   к колонкам train через `reindex`.

Новая категория не добавляет столбцов: соответствующие ей one-hot значения
будут нулевыми. Все строки и их порядок сохраняются.
Train непустой; для пустой новой таблицы возврат уже предусмотрен.

**На validation не вызывайте `fit` и не пересчитывайте медианы.**
Используйте состояние, полученное только по train.

In [ ]:
from sklearn.preprocessing import StandardScaler


def fit_preprocessor(X):
    numeric = X.select_dtypes('number').columns.tolist()
    categorical = [col for col in X.columns if col not in numeric]

    medians = None  # TODO: медианы числовых колонок; NaN замените на 0.
    numbers = X[numeric].fillna(medians)
    scaler = None
    if numeric:
        scaler = None  # TODO: обучите StandardScaler на numbers.

    frame = X.copy()
    frame[numeric] = numbers
    frame[categorical] = frame[categorical].astype(object).fillna('__MISSING__')
    columns = pd.get_dummies(frame, columns=categorical, dtype=float).columns.tolist()
    return {
        'input_columns': X.columns.tolist(),
        'numeric': numeric,
        'categorical': categorical,
        'medians': medians,
        'scaler': scaler,
        'columns': columns,
    }


def transform_preprocessor(X, state):
    if len(X) == 0:
        return pd.DataFrame(index=X.index, columns=state['columns'], dtype=float)
    frame = X[state['input_columns']].copy()
    numeric, categorical = state['numeric'], state['categorical']
    if numeric:
        numbers = frame[numeric].fillna(state['medians'])
        frame[numeric] = state['scaler'].transform(numbers)
    frame[categorical] = frame[categorical].astype(object).fillna('__MISSING__')
    encoded = pd.get_dummies(frame, columns=categorical, dtype=float)
    return encoded.reindex(columns=state['columns'], fill_value=0)

In [ ]:
train = pd.DataFrame({'value': [1., 3., np.nan], 'city': ['a', 'b', np.nan]})
state = fit_preprocessor(train)
assert state['medians']['value'] == 2
assert isinstance(state['scaler'], StandardScaler)
assert np.allclose(state['scaler'].scale_, [np.sqrt(2/3)])  # ddof=0.
assert state['columns'] == ['value', 'city___MISSING__', 'city_a', 'city_b']

new = pd.DataFrame({'value': [2., np.nan], 'city': ['unknown', 'a']}, index=[8, 4])
prepared = transform_preprocessor(new, state)
assert np.allclose(prepared['value'], [0., 0.])
assert prepared.index.equals(new.index) and prepared.iloc[0].eq(0).all()
display(prepared)

In [ ]:
preprocessing_state = fit_preprocessor(X_adult)
prepared_adult = transform_preprocessor(X_adult, preprocessing_state)
pd.Series({
    'строк': len(prepared_adult),
    'признаков': prepared_adult.shape[1],
    'пропусков': prepared_adult.isna().sum().sum(),
})

### [G. Отберём признаки с помощью SelectKBest](https://contest.yandex.ru/contest/101854/problems/G/)

После one-hot колонок становится много. Попробуем оставить десять.
`SelectKBest` с `f_classif` оценивает каждый признак отдельно:
сравнивает различие средних в двух классах с разбросом внутри классов.

В контест отправьте две функции:

- `fit_selector(X_train, y_train, k=10)` обучает
  `SelectKBest(score_func=f_classif, k=k)` и возвращает словарь:
  `input_columns` — список исходных колонок по порядку,
  `selector` — обученный объект отбора.
- `transform_selector(X, state)` возвращает numpy-массив с выбранными
  колонками в порядке строк X. Перед применением выберите входные колонки
  в сохранённом порядке. Для пустого X верните массив формы `(0, k)`.

X — уже подготовленные числа без пропусков; в train есть оба класса.
`k` — целое от 1 до числа колонок. Исходные данные и state не меняйте.
Отбор обучайте только на train: ему нужен target.

**Подсказки:** `SelectKBest(score_func=f_classif, k=k)`, `.fit(X_train, y_train)`,
`.transform(X)` и `.get_support()`. Этот метод не использует случайность.
После функций дан готовый код сравнения модели на всех признаках
и на выбранных десяти.


In [ ]:
selection_preprocessing = fit_preprocessor(X_adult)
X_select_train = transform_preprocessor(X_adult, selection_preprocessing)
X_select_valid = transform_preprocessor(X_adult_valid, selection_preprocessing)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif


def fit_selector(X_train, y_train, k=10):
    # TODO: создайте и обучите SelectKBest.
    return {
        'input_columns': X_train.columns.tolist(),
        'selector': None,
    }


def transform_selector(X, state):
    if len(X) == 0:
        return np.empty((0, int(state['selector'].get_support().sum())), dtype=float)
    # TODO: выберите колонки в сохранённом порядке и примените отбор.
    return None


In [ ]:
sample = pd.DataFrame({'signal': [0., 1., 2., 8., 9., 10.],
                       'noise': [1., 4., 2., 3., 1., 4.]}, index=[6, 2, 5, 1, 4, 3])
labels = pd.Series([0, 0, 0, 1, 1, 1], index=sample.index)
selection = fit_selector(sample, labels, k=1)
assert np.allclose(transform_selector(sample[['noise', 'signal']], selection), sample[['signal']].to_numpy())
assert transform_selector(sample.iloc[:0], selection).shape == (0, 1)

selection_state = fit_selector(X_select_train, y_adult, k=10)
selector = selection_state['selector']
selected_train = transform_selector(X_select_train, selection_state)
selected_valid = transform_selector(X_select_valid, selection_state)
selected_columns = X_select_train.columns[selector.get_support()].tolist()
pd.Series(selected_columns, name='выбранный признак')


In [ ]:
from sklearn.linear_model import LogisticRegression

selection_all_model = LogisticRegression(max_iter=1000, random_state=42)
selection_all_model.fit(X_select_train, y_adult)
selection_all_proba = selection_all_model.predict_proba(X_select_valid)[:, 1]
selection_all_auc = roc_auc_score(y_adult_valid, selection_all_proba)

selection_top10_model = LogisticRegression(max_iter=1000, random_state=42)
selection_top10_model.fit(selected_train, y_adult)
selection_top10_proba = selection_top10_model.predict_proba(selected_valid)[:, 1]
selection_top10_auc = roc_auc_score(y_adult_valid, selection_top10_proba)

selection_comparison = pd.DataFrame({
    'вариант': ['Все подготовленные признаки', 'SelectKBest: 10 признаков'],
    'число признаков': [X_select_train.shape[1], selected_train.shape[1]],
    'validation_ROC_AUC': [selection_all_auc, selection_top10_auc],
})
selection_comparison

Какие признаки остались? Как изменился ROC-AUC?
Почему уменьшение числа признаков не обязательно улучшает результат?
Учтите, что метод рассматривает признаки по одному и не проверяет их сочетания.

**Ваш ответ:** ...

### [H. Соберём и отправим свою модель](https://contest.yandex.ru/contest/101854/problems/H/)

Теперь соберите подготовку данных и обучение в одно решение.
В контест нужно будет отправить **код функций**, которые получают исходные
таблицы Adult. Все преобразования должны происходить внутри этих функций.

Ваша модель должна использовать два признака из B и ограничение
`hours_per_week` по квантилям 0.01 и 0.99 из C. Для заполнения пропусков,
масштабирования и one-hot используйте функции из F.
Модель — `LogisticRegression(max_iter=1000, random_state=42)`.

**Шаг 1. Получите ориентир для сравнения.**

Готовый код ниже обучает baseline на исходных признаках с подготовкой из F.
Затем сравним его с вашей полной цепочкой на тех же строках validation.


In [ ]:
from sklearn.linear_model import LogisticRegression

baseline_preprocessing = fit_preprocessor(X_adult)
X_baseline_train = transform_preprocessor(X_adult, baseline_preprocessing)
X_baseline_valid = transform_preprocessor(X_adult_valid, baseline_preprocessing)

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_baseline_train, y_adult)
baseline_proba = baseline_model.predict_proba(X_baseline_valid)[:, 1]
baseline_auc = roc_auc_score(y_adult_valid, baseline_proba)
pd.Series({'baseline_ROC_AUC': baseline_auc})


**Шаг 2. Напишите две функции.**

**`fit_model(X_train, y_train)`**

1. Добавьте признаки из B к копии исходного train.
2. Найдите границы для `hours_per_week` по train и примените их.
3. Обучите подготовку из F и преобразуйте полученную таблицу.
4. Создайте и обучите логистическую регрессию.

Верните словарь с тремя ключами:

- `bounds` — границы для `hours_per_week` из C;
- `preprocessing` — состояние подготовки из F;
- `model` — обученная `LogisticRegression`.

**`predict_proba(state, X)`**

Добавьте те же признаки, примените сохранённые границы и подготовку,
затем получите вероятности класса 1. Верните одномерный вещественный
`numpy.ndarray` формы `(len(X),)` в порядке строк `X`.
Для пустого `X` верните пустой массив.

Исходные таблицы не меняйте. При предсказании ничего не обучайте заново:
все нужные настройки уже находятся в `state`.
Функции должны работать без переменных вроде `X_adult` и `X_clip_train`.

В `X_train` и `X` есть исходные колонки Adult, включая `capital_gain` и
`hours_per_week`; уже созданные признаки им не передаются.
Train непустой, в `y_train` есть оба класса.
Пропуски и неизвестные категории обработают ваши функции из F.


In [ ]:
def fit_model(X_train, y_train):
    # TODO: добавьте признаки, найдите и примените границы, подготовьте train.
    # TODO: создайте LogisticRegression с указанными параметрами и обучите её.
    return {
        'bounds': None,
        'preprocessing': None,
        'model': None,
    }


def predict_proba(state, X):
    if len(X) == 0:
        return np.empty(0, dtype=float)

    # TODO: повторите преобразования с готовыми настройками и получите вероятности.
    return None


Самопроверка: пропуски, новая категория и пустая таблица.


In [ ]:
sample = pd.DataFrame({
    'age': [20, 25, 30, 35, 40, 45],
    'capital_gain': [0., 0., 2., np.nan, 10., 5.],
    'hours_per_week': [20., 30., 40., 40., 50., 99.],
    'occupation': ['a', 'a', 'b', None, 'a', 'b'],
}, index=[10, 30, 20, 60, 40, 50])
labels = pd.Series([0, 0, 0, 1, 1, 1], index=sample.index)
state = fit_model(sample, labels)
new_rows = sample.iloc[:2].copy()
new_rows.loc[10, 'occupation'] = 'new'
proba = predict_proba(state, new_rows)
assert isinstance(proba, np.ndarray) and proba.shape == (2,)
assert np.isfinite(proba).all() and ((proba >= 0) & (proba <= 1)).all()
assert predict_proba(state, sample.iloc[:0]).shape == (0,)


**Шаг 3. Сравните качество с baseline.**

Передавайте в функции исходные train и validation: добавление признаков и
остальная подготовка теперь находятся внутри вашего решения.
Готовая ячейка покажет ROC-AUC двух вариантов.
Рост качества не гарантирован — проверьте результат на validation,
не меняя разбиение и не обучаясь на этой части.


In [ ]:
model_state = fit_model(X_adult, y_adult)
valid_proba = predict_proba(model_state, X_adult_valid)
model_auc = roc_auc_score(y_adult_valid, valid_proba)

comparison = pd.DataFrame({
    'вариант': ['Baseline: подготовка из F', 'Моя модель: B + C + F'],
    'validation_ROC_AUC': [baseline_auc, model_auc],
})
comparison['разница_с_baseline'] = comparison['validation_ROC_AUC'] - baseline_auc
comparison


**Шаг 4. Подготовьте код к отправке.**

Соберите один файл `.py` с импортами и функциями:

- `fit_model`, `predict_proba` — из H;
- `add_adult_features` — из B;
- `fit_clip_bounds`, `apply_clip` — из C;
- `fit_preprocessor`, `transform_preprocessor` — из F.

Разрешённые модули: `numpy`, `pandas`, `sklearn.preprocessing`,
`sklearn.linear_model`. Модель — только `LogisticRegression`.
Не включайте загрузку CSV, запуск экспериментов, графики и проверки `assert`.
Функции получают данные аргументами и возвращают результат через `return`.

Проверяющая система сама вызовет `fit_model` на train, затем `predict_proba`
на исходных строках отложенного test. Она проверит формат вероятностей,
работу преобразований и рассчитает ROC-AUC. Выводы из ноутбука отправлять не нужно.

Перед отправкой убедитесь, что новые признаки создаются и при обучении,
и при предсказании, а границы и настройки подготовки берутся только из train.
